# Phase 5 — Validation de la construction des portefeuilles

Ce notebook vérifie les CSV. Il ne documente pas les stratégies et il ne commente pas les écarts.

- Documentation des stratégies : `documentation/05_portfolio_construction.ipynb`
- Calculs détaillés : `reports/portfolio/05_calculs_detailles.ipynb`
- Analyse comparative : `notebooks/05_analyse_comparative.ipynb`

La vérification échoue si les stratégies ne partagent pas les mêmes titres et les mêmes mois, si un poids est négatif ou si la somme s'écarte de 1, si l'équipondéré n'est pas exactement égal, si le SARON mensuel ne correspond pas au taux annuel, si une métrique publiée s'écarte du recalcul, si la volatilité minimale n'est pas la plus basse, ou si le Sharpe maximal n'est pas le plus haut.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, REPORTING_CURRENCY
from src.portfolio_construction.build_portfolios import (
    COMPARISON_FILE,
    MEMBERSHIP_FILE,
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    STRATEGY_IDS,
    WEIGHTS_FILE,
    build_portfolios,
    construct_portfolios,
    load_construction_inputs,
    measure_strategy,
    portfolio_return,
)
from src.preparation.quality_treatment import CHF_UNIVERSE, SARON_TREATED

build_portfolios()
prices, universe, saron = load_construction_inputs()
constructed = construct_portfolios(prices, universe, saron)

weights = pd.read_csv(WEIGHTS_FILE)
metrics = pd.read_csv(METRICS_FILE)
comparison = pd.read_csv(COMPARISON_FILE)
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
membership = pd.read_csv(MEMBERSHIP_FILE, dtype=str, keep_default_na=False)
saron_file = pd.read_csv(SARON_TREATED)

assert list(weights["strategy"].unique()) == list(STRATEGY_IDS)
assert set(metrics["strategy"]) == set(STRATEGY_IDS)
assert set(monthly["strategy"]) == set(STRATEGY_IDS)
assert len(membership) == len(universe)
assert membership["yahoo_symbol"].is_unique

included = set(membership.loc[membership["decision"].eq("included"), "yahoo_symbol"])
assert len(included) >= 2
for strategy, group in weights.groupby("strategy"):
    assert set(group["yahoo_symbol"]) == included
    assert abs(group["weight"].sum() - 1) < 1e-8
    assert group["weight"].min() >= -1e-12
    assert group["yahoo_symbol"].is_unique

equal = weights.loc[weights["strategy"].eq("equal_weight"), "weight"]
assert np.allclose(equal.to_numpy(), 1 / len(included))

excluded = set(membership.loc[membership["decision"].eq("excluded"), "yahoo_symbol"])
assert included.isdisjoint(excluded)
assert not set(weights["yahoo_symbol"]).intersection(excluded)

for reason in ("no_snb_series", "no_price_history", "incomplete_monthly_history"):
    blocked = set(membership.loc[membership["reason"].eq(reason), "yahoo_symbol"])
    assert blocked.isdisjoint(set(weights["yahoo_symbol"]))

assert metrics["months"].eq(MARKET_HISTORY_YEARS * 12).all()
assert metrics["names"].eq(len(included)).all()
assert metrics["window_start"].nunique() == 1
assert metrics["window_end"].nunique() == 1
assert set(metrics["risk_free_series"]) == {"SARON"}
assert set(metrics["risk_free_currency"]) == {REPORTING_CURRENCY}
assert set(saron_file["rate_currency"]) == {REPORTING_CURRENCY}
assert set(saron_file["series_id"]) == {"SARON"}

saron_file["risk_free_return"] = (1 + saron_file["value"].astype(float) / 100) ** (1 / 12) - 1
checked = monthly.merge(
    saron_file.loc[:, ["date", "risk_free_return"]],
    left_on="month",
    right_on="date",
    suffixes=("", "_saron"),
)
assert len(checked) == len(monthly)
assert np.allclose(checked["risk_free_return"], checked["risk_free_return_saron"])
assert np.allclose(
    checked["excess_return"],
    checked["portfolio_return"] - checked["risk_free_return"],
)

asset_returns = constructed["asset_returns"]
risk_free = constructed["risk_free"]
for strategy in STRATEGY_IDS:
    stored_weights = weights.loc[weights["strategy"].eq(strategy)].set_index("yahoo_symbol")["weight"]
    stored_weights = stored_weights.reindex(asset_returns.columns)
    returns = portfolio_return(asset_returns, stored_weights)
    saved = monthly.loc[monthly["strategy"].eq(strategy)].set_index("month").loc[returns.index]
    assert np.allclose(saved["portfolio_return"], returns.to_numpy())
    measured = measure_strategy(strategy, asset_returns, risk_free, stored_weights)
    published = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    for column in (
        "annualized_return",
        "annualized_volatility",
        "sharpe_ratio",
        "maximum_drawdown",
        "maximum_weight",
        "effective_holdings",
    ):
        assert abs(measured[column] - float(published[column])) < 1e-8

    excess = returns - risk_free.reindex(returns.index)
    sharpe = float(excess.mean() / excess.std(ddof=1) * np.sqrt(12))
    growth = float(np.prod(1 + returns.to_numpy()))
    annualized = growth ** (12 / len(returns)) - 1
    volatility = float(returns.std(ddof=1) * np.sqrt(12))
    wealth = (1 + returns).cumprod()
    drawdown = float((wealth / wealth.cummax() - 1).min())
    assert abs(sharpe - float(published["sharpe_ratio"])) < 1e-8
    assert abs(annualized - float(published["annualized_return"])) < 1e-8
    assert abs(volatility - float(published["annualized_volatility"])) < 1e-8
    assert abs(drawdown - float(published["maximum_drawdown"])) < 1e-8

ordered = metrics.set_index("strategy")
assert ordered.loc["minimum_volatility", "annualized_volatility"] <= ordered["annualized_volatility"].min() + 1e-8
assert ordered.loc["maximum_sharpe", "sharpe_ratio"] >= ordered["sharpe_ratio"].max() - 1e-8

for _, row in comparison.iterrows():
    equal_value = float(row["equal_weight"])
    minimum_value = float(row["minimum_volatility"])
    sharpe_value = float(row["maximum_sharpe"])
    assert abs(minimum_value - equal_value - float(row["minimum_volatility_minus_equal_weight"])) < 1e-10
    assert abs(sharpe_value - equal_value - float(row["maximum_sharpe_minus_equal_weight"])) < 1e-10
    assert abs(equal_value - float(ordered.loc["equal_weight", row["metric"]])) < 1e-10

fresh = constructed["metrics"].set_index("strategy")
for strategy in STRATEGY_IDS:
    for column in ("annualized_return", "annualized_volatility", "sharpe_ratio"):
        assert abs(float(fresh.loc[strategy, column]) - float(ordered.loc[strategy, column])) < 1e-8

print(
    f"Titres communs : {len(included)}. "
    f"Mois : {int(ordered['months'].iloc[0])}. "
    f"Fenêtre : {ordered['window_start'].iloc[0]} à {ordered['window_end'].iloc[0]}."
)
print("Résultat : PASS")

Titres communs : 83. Mois : 120. Fenêtre : 2016-10 à 2026-09.
Résultat : PASS
